# **Importamos las Librerias**


In [1]:
import os
import torch
import numpy as np
import torch.nn as nn
import pandas as pd
import torch.optim as optim
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import confusion_matrix
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
import random
import joblib  # opcional: para guardar scaler, etc.

# (opcional) si NO vas a re-hacer logística por scipy, puedes omitir:
# from scipy import optimize



ModuleNotFoundError: No module named 'torch'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **Cargar el CSV y revisar rápido.**

In [ ]:

file_path = "/content/drive/MyDrive/Programas de IA/Laboratorios/laboratorio5/covtype.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"No se encontró el archivo en: {file_path}")

data = pd.read_csv(file_path)

# Chequeos iniciales
print("Shape:", data.shape)       # filas y columnas
print("Columnas:", data.columns[:10], "...")  # mostrar primeras columnas
print("\nPrimeras filas:")
print(data.head())

print("\nInformación del dataset:")
print(data.info())

# Verificar que exista la columna objetivo
if "Cover_Type" in data.columns:
    print("\nDistribución de la variable objetivo (Cover_Type):")
    print(data["Cover_Type"].value_counts())
else:
    print("⚠️ No se encontró la columna 'Cover_Type'. Revisa los nombres de columna.")

sas

In [ ]:

# Detectar columna objetivo disponible
possible_targets = ["Class_num", "Class", "Cover_Type", "cover_type", "target", "label"]
TARGET = None
for c in possible_targets:
    if c in data.columns:
        TARGET = c
        break
if TARGET is None:
    raise ValueError(f"No encontré la columna objetivo. Columnas: {list(data.columns)[:10]} ...")

print("Usando como target:", TARGET)

# Separar X e y (si existe una columna de texto 'Class', la excluimos de X)
cols_to_drop = [TARGET]
if "Class" in data.columns and "Class" != TARGET:
    cols_to_drop.append("Class")

X = data.drop(columns=cols_to_drop).values
y_raw = data[TARGET].values

# Asegurar etiquetas 0..K-1
# (Cover_Type suele venir 1..7; esto lo convierte a 0..6)
le = LabelEncoder()
y = le.fit_transform(y_raw)

print("Shape X:", X.shape, "| y:", y.shape)
print("Clases originales:", sorted(pd.unique(y_raw))[:10], "...")
print("Clases mapeadas 0..K-1:", list(range(len(le.classes_))))


division

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # mantiene distribución de clases
)

print("Tamaño train:", X_train.shape, y_train.shape)
print("Tamaño test:", X_test.shape, y_test.shape)

# **Nomalizacion**

In [ ]:
mu = np.mean(X_train, axis=0)
sigma = np.std(X_train, axis=0)

X_train_norm = (X_train - mu) / sigma
X_test_norm = (X_test - mu) / sigma

# **Coventir a tensores**

In [ ]:
X_train_tensor = torch.from_numpy(X_train_norm).float()
y_train_tensor = torch.from_numpy(y_train).long()
X_test_tensor = torch.from_numpy(X_test_norm).float()
y_test_tensor = torch.from_numpy(y_test).long()

In [ ]:
# --- 1) DataLoader (minibatches) ---
from torch.utils.data import TensorDataset, DataLoader

batch_size = 256
train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds  = TensorDataset(X_test_tensor,  y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=1024, shuffle=False)

# --- 2) Dispositivo ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train_tensor = X_train_tensor.to(device); y_train_tensor = y_train_tensor.to(device)
X_test_tensor  = X_test_tensor.to(device);  y_test_tensor  = y_test_tensor.to(device)

# --- 3) Modelo (el tuyo) + optimizador Adam ---
class ModelCustom2(nn.Module):
    def __init__(self, D_in, H, D_out):
        super().__init__()
        self.fc1 = nn.Linear(D_in, H)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(H, D_out)
    def forward(self, x):
        x1 = self.fc1(x)
        x  = self.relu(x1)
        x  = self.fc2(x + x1)  # tu residual
        return x

D_in = X_train_tensor.shape[1]
D_out = int(y_train_tensor.max().item() + 1)

model = ModelCustom2(D_in=D_in, H=100, D_out=D_out).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)  # más estable

# --- 4) Entrenamiento con minibatches + logging de acc ---

epochs   = 100
log_each = 10

def accuracy(loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(device))
            pred = logits.argmax(1)
            correct += (pred == yb.to(device)).sum().item()
            total   += yb.size(0)
    return correct / total

model.train()
for e in range(1, epochs+1):
    epoch_losses = []
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())

    if e % log_each == 0 or e == 1:
        tr_acc = accuracy(train_loader)
        te_acc = accuracy(test_loader)
        print(f"Epoch {e:3d}/{epochs}  loss={np.mean(epoch_losses):.4f}  "
              f"train_acc={tr_acc:.4f}  test_acc={te_acc:.4f}")

# evaluación final
final_test_acc = accuracy(test_loader)
print(f"\nAccuracy FINAL en test: {final_test_acc:.4f}")


In [ ]:
model.eval()
with torch.no_grad():
    logits = model(X_test_tensor)
    y_pred_labels = torch.argmax(logits, dim=1).cpu().numpy()

cm = confusion_matrix(y_test, y_pred_labels)

print("Matriz de confusión (valores):\n", cm)

# Visualización rápida
fig, ax = plt.subplots(figsize=(6,6))
im = ax.imshow(cm, interpolation='nearest')
ax.figure.colorbar(im, ax=ax)
ax.set_xlabel('Predicha'); ax.set_ylabel('Real')
ax.set_title('Matriz de Confusión')
plt.show()


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_labels, digits=4))


In [ ]:
from sklearn.metrics import accuracy_score

ACC_NN = accuracy_score(y_test, y_pred_labels)
ACC_OVA = 0.6411   # <-- reemplaza con tu accuracy del modelo One-vs-All

delta = ACC_NN - ACC_OVA
print(f"Accuracy One-vs-All: {ACC_OVA:.4f}")
print(f"Accuracy Red Neuronal: {ACC_NN:.4f}")
print(f"Mejora absoluta: {delta:+.4f}")
if ACC_OVA > 0:
    print(f"Mejora relativa: {delta/ACC_OVA*100:+.2f}%")


# **Conclusiones**
Se implementó una **red neuronal multicapa (MLP)** con PyTorch utilizando el dataset covtype.csv.

El modelo alcanzó una precisión en test de ≈ 84%, lo que demuestra un buen poder de generalización.

La red neuronal **supera** al modelo One-vs-All (regresión logística multiclase) en** términos de exactitud**, mostrando que la capacidad de representación no lineal de la red permite capturar mejor las relaciones en los datos.

La **normalización** de los datos y la correcta división estratificada del dataset fueron pasos **clave** para obtener un entrenamiento estable.

Los resultados de la matriz de confusión y el reporte de clasificación confirman que,** aunque existen clases más difíciles** de distinguir, la red mejora significativamente en la mayoría de las categorías.